In [0]:
from pyspark.sql import functions as F

STORAGE = "stolistdl2026"
BRONZE = f"abfss://bronze@{STORAGE}.dfs.core.windows.net/olist/"
CATALOG, SCHEMA = "olist_dev", "silver"

def read_bronze(filename):
    """Bronze CSV padho. Sab columns string rahenge, types hum khud lagayenge."""
    return (spark.read
        .option("header", True)
        .option("multiLine", True)
        .option("quote", '"')
        .option("escape", '"')
        .csv(BRONZE + filename)
        .withColumn("source_file", F.col("_metadata.file_path")))

def clean_strings(df):
    """Har string column ko trim karo, aur khali string ko NULL banao."""
    for c, t in df.dtypes:
        if t == "string" and c != "source_file":
            df = df.withColumn(c, F.when(F.trim(F.col(c)) == "", None).otherwise(F.trim(F.col(c))))
    return df

def apply_types(df, casts):
    for col, dtype in casts.items():
        if dtype == "timestamp":
            df = df.withColumn(col, F.to_timestamp(F.col(col)))
        else:
            df = df.withColumn(col, F.col(col).cast(dtype))
    return df

def write_silver(df, table):
    df = df.withColumn("ingestion_ts", F.current_timestamp())
    (df.write.format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(f"{CATALOG}.{SCHEMA}.{table}"))
    print(f"{table}: {spark.table(f'{CATALOG}.{SCHEMA}.{table}').count()} rows")

In [0]:
TABLES = {
    "customers": {
        "file": "olist_customers_dataset.csv",
        "keys": ["customer_id"],
        "casts": {}
    },
    "orders": {
        "file": "olist_orders_dataset.csv",
        "keys": ["order_id"],
        "casts": {c: "timestamp" for c in [
            "order_purchase_timestamp", "order_approved_at",
            "order_delivered_carrier_date", "order_delivered_customer_date",
            "order_estimated_delivery_date"]}
    },
    "order_items": {
        "file": "olist_order_items_dataset.csv",
        "keys": ["order_id", "order_item_id"],
        "casts": {"order_item_id": "int", "shipping_limit_date": "timestamp",
                  "price": "double", "freight_value": "double"}
    },
    "order_payments": {
        "file": "olist_order_payments_dataset.csv",
        "keys": ["order_id", "payment_sequential"],
        "casts": {"payment_sequential": "int", "payment_installments": "int",
                  "payment_value": "double"}
    },
    "order_reviews": {
        "file": "olist_order_reviews_dataset.csv",
        "keys": ["review_id", "order_id"],
        "casts": {"review_score": "int", "review_creation_date": "timestamp",
                  "review_answer_timestamp": "timestamp"}
    },
    "products": {
        "file": "olist_products_dataset.csv",
        "keys": ["product_id"],
        "casts": {"product_name_length": "int", "product_description_length": "int",
                  "product_photos_qty": "int", "product_weight_g": "double",
                  "product_length_cm": "double", "product_height_cm": "double",
                  "product_width_cm": "double"}
    },
    "sellers": {
        "file": "olist_sellers_dataset.csv",
        "keys": ["seller_id"],
        "casts": {}
    },
    "category_translation": {
        "file": "product_category_name_translation.csv",
        "keys": ["product_category_name"],
        "casts": {}
    },
}

In [0]:
for table, cfg in TABLES.items():
    df = read_bronze(cfg["file"])

    # Products file mein original dataset ki spelling galti hai ("lenght"), usse theek karo
    if table == "products":
        df = (df.withColumnRenamed("product_name_lenght", "product_name_length")
                .withColumnRenamed("product_description_lenght", "product_description_length"))

    df = clean_strings(df)
    df = apply_types(df, cfg["casts"])
    df = df.dropDuplicates(cfg["keys"])
    write_silver(df, table)

customers: 99441 rows
orders: 99441 rows
order_items: 112650 rows
order_payments: 103886 rows
order_reviews: 99224 rows
products: 32951 rows
sellers: 3095 rows
category_translation: 71 rows


In [0]:
geo = read_bronze("olist_geolocation_dataset.csv")
geo = clean_strings(geo)
geo = (geo
    .withColumn("geolocation_lat", F.col("geolocation_lat").cast("double"))
    .withColumn("geolocation_lng", F.col("geolocation_lng").cast("double"))
    .groupBy("geolocation_zip_code_prefix")
    .agg(F.avg("geolocation_lat").alias("lat"),
         F.avg("geolocation_lng").alias("lng"),
         F.first("geolocation_city", ignorenulls=True).alias("city"),
         F.first("geolocation_state", ignorenulls=True).alias("state"),
         F.first("source_file").alias("source_file")))
write_silver(geo, "geolocation")

geolocation: 19015 rows


In [0]:
display(spark.sql("SHOW TABLES IN olist_dev.silver"))
spark.table("olist_dev.silver.orders").printSchema()
display(spark.table("olist_dev.silver.orders").limit(5))

database,tableName,isTemporary
silver,category_translation,false
silver,customers,false
silver,geolocation,false
silver,order_items,false
silver,order_payments,false
silver,order_reviews,false
silver,orders,false
silver,products,false
silver,sellers,false


root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: timestamp (nullable = true)
 |-- source_file: string (nullable = true)
 |-- ingestion_ts: timestamp (nullable = true)



order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,source_file,ingestion_ts
2807d0e504d6d4894d41672727bc139f,72ae281627a6102d9b3718528b420f8a,delivered,2018-02-03T20:37:35.000Z,2018-02-03T20:50:22.000Z,2018-02-05T22:37:28.000Z,2018-02-08T16:13:46.000Z,2018-02-21T00:00:00.000Z,abfss://bronze@stolistdl2026.dfs.core.windows.net/olist/olist_orders_dataset.csv,2026-10-14T19:09:11.567Z
ccbabeb0b02433bd0fcbac46e70339f2,c77ee2d8ba1614a4d489a44166894938,delivered,2018-02-19T20:31:09.000Z,2018-02-21T06:15:25.000Z,2018-02-22T21:04:23.000Z,2018-03-09T22:22:25.000Z,2018-03-13T00:00:00.000Z,abfss://bronze@stolistdl2026.dfs.core.windows.net/olist/olist_orders_dataset.csv,2026-10-14T19:09:11.567Z
e4de6d53ecff736bc68804b0b6e9f635,9f6618c17568ac301465fe7ad056c674,delivered,2017-10-16T14:56:50.000Z,2017-10-17T03:49:34.000Z,2017-10-27T22:14:21.000Z,2017-11-08T21:25:24.000Z,2017-11-21T00:00:00.000Z,abfss://bronze@stolistdl2026.dfs.core.windows.net/olist/olist_orders_dataset.csv,2026-10-14T19:09:11.567Z
cadbb3657dac2dbbd5b84b12e7b78aad,93ada7a24817edda9f4ab998fa823d16,delivered,2018-02-27T12:55:42.000Z,2018-03-01T02:48:54.000Z,2018-03-03T02:27:03.000Z,2018-03-16T14:59:01.000Z,2018-03-29T00:00:00.000Z,abfss://bronze@stolistdl2026.dfs.core.windows.net/olist/olist_orders_dataset.csv,2026-10-14T19:09:11.567Z
d3d6788577c9592da441752e8a1dd5e3,8628fac2267e8c8804525da99c10ed0e,delivered,2017-09-19T22:17:15.000Z,2017-09-20T07:55:14.000Z,2017-09-22T17:23:09.000Z,2017-10-10T18:43:53.000Z,2017-10-13T00:00:00.000Z,abfss://bronze@stolistdl2026.dfs.core.windows.net/olist/olist_orders_dataset.csv,2026-10-14T19:09:11.567Z


In [0]:
%sql
DESCRIBE HISTORY olist_dev.silver.customers;

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
0,2026-10-14T19:09:08.000Z,148517414416831,ankitsaxena386@gmail.com,CREATE OR REPLACE TABLE AS SELECT,"Map(isV1SaveAsTableOverwrite -> true, partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, canOverwriteSchema -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""databricks.internal.autoUpgrades.delta.parquet.format.version.afe.internal"":""2.12.0"",""delta.parquet.format.version.afe.internal"":""2.12.0"",""delta.enableDeletionVectors"":""true"",""databricks.internal.autoUpgrades.delta.parquet.format.version"":""2.12.0"",""delta.parquet.format.version"":""2.12.0"",""io.unitycatalog.tableId"":""2032d777-f581-48f4-8e0b-18150df8c14e""}, statsOnLoad -> true)",null,List(1708551959728824),10d5f4b9-f6b8-4e26-a77e-b5aaba5dd69d,1014-182238-3zx8rzqi-v2n,null,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 0, numRemovedBytes -> 0, numDeletionVectorsRemoved -> 0, numOutputRows -> 99441, numOutputBytes -> 4065245)",null,Databricks-Runtime/19.9.x-aarch64-photon-scala2.13


In [0]:
%sql
SELECT COUNT(*) FROM olist_dev.silver.customers VERSION AS OF 0;

COUNT(*)
99441
